In [9]:
import os
import time

import cv2
import pandas as pd
import torch
from natsort import natsorted
from ultralytics import YOLO

from code_programm.path import get_path_weight_model

In [10]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
model_speed = YOLO(get_path_weight_model('speed_recognition.pt'))

1
NVIDIA GeForce GTX 1080 Ti


In [11]:
wheel_ets_train_speed = pd.DataFrame(columns=['speed'])

In [12]:
paths = [r'D:\Dataset_for_autopilot\2024-03-31 03-29-46',
         r'D:\Dataset_for_autopilot\2024-03-31 03-28-05',
         r'D:\Dataset_for_autopilot\2024-03-31 03-26-14',
         r'D:\Dataset_for_autopilot\2024-03-31 03-24-22',
         r'D:\Dataset_for_autopilot\2024-03-31 03-22-07',
         r'D:\Dataset_for_autopilot\2024-03-31 03-17-11',
         r'D:\Dataset_for_autopilot\2024-03-31 03-11-36',
         r'D:\Dataset_for_autopilot\2024-03-31 03-05-57', ]

In [13]:
for j in paths:
    path_i = os.path.join(j, f'speed')
    if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
        png_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
        print("Полные пути к файлам в папке:")
        png_files = natsorted(png_files)
        print(png_files[0])
    else:
        print("Указанный путь не существует или не является папкой.")
    start_time = time.time()
    for file in png_files:
        image = cv2.imread(file)
        new_image = cv2.resize(image, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
        results = model_speed.predict(new_image, conf=0.9, device='cuda', verbose=False, show=False)
        sorted_objects = sorted(
            ({'class': int(cls), 'confidence': float(conf), 'xmin': int(xmin), 'ymin': int(ymin), 'xmax': int(xmax),
              'ymax': int(ymax)}
             for result in results for obj in result.boxes.data for xmin, ymin, xmax, ymax, conf, cls in
             (obj.tolist(),)),
            key=lambda obj: obj['xmin']
        )
        if sorted_objects:
            speed = ''.join(str(obj['class']) for obj in sorted_objects)
        else:
            speed = ''
        name = file.split(f'\\')[-1].replace('.png', '')
        wheel_ets_train_speed.loc[len(wheel_ets_train_speed)] = speed
    print(len(wheel_ets_train_speed), 'сек:', time.time() - start_time)

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-03-31 03-29-46\speed\2024-03-31 03-29-46_0.png
566 сек: 7.953060865402222
Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-03-31 03-28-05\speed\2024-03-31 03-28-05_0.png
1253 сек: 8.628629207611084
Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-03-31 03-26-14\speed\2024-03-31 03-26-14_0.png
1942 сек: 8.65347409248352
Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-03-31 03-24-22\speed\2024-03-31 03-24-22_0.png
2305 сек: 4.5663182735443115
Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-03-31 03-22-07\speed\2024-03-31 03-22-07_0.png
3183 сек: 11.029048204421997
Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-03-31 03-17-11\speed\2024-03-31 03-17-11_0.png
4856 сек: 21.45240879058838
Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-03-31 03-11-36\speed\2024-03-31 03-11-36_0.png
7289 сек: 31.082301378250122
Полные пути к файлам в папке:
D:\Dataset_for_autopilot

In [14]:
len(wheel_ets_train_speed)

7861

In [15]:
wheel_ets_train_speed.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed.csv', index=False)

In [16]:
wheel_ets_train_speed_ = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed.csv')
len(wheel_ets_train_speed_)

7861